# Merging Data - Radon Risk Prediction with Surficial Geology Features at FSA Level

##### **What was done in this notebook:**

**1. Clean the surficial geology dataset**

Remove unnecessary columns and remained only "UTYPE1", "ULABEL1", "HYDRO_INT", and "geometry". The major sediment category from "UTYPE1" was extracted as a new variable, "major_sedi_type".

**2. Join FSA data**

Since more than 50% of FSAs contain multiple sedimentary types, a simple spatial join would not adequately represent geological composition.

Instead, I computed the area-weighted percentage of each major sedimentary type (14 in total) within each FSA.

Note: geometries were reprojected to **EPSG:3347 (meters)** before computing areas.

**3. Merged radon survey data**

Performed an inner join between FSA-level surficial features and radon survey records, keeping only valid FSA matches (247 unmatched entries were removed).

Radon concentrations reported as "<15" were replaced with 7.5 for numerical consistency.

In [1]:
import geopandas as gpd
import fiona
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
##### CHANGE DIRECTORIES #####
#surficial_path = Path("/Path/to/the/shpgeo/directory")
#fsa_path = Path("/Path/to/fsa/boundary/dataset/gfsa000a11a_e.shp")
#radon_path = Path("/Path/to/cross-Canada/radon/survey/dataset/radon-concentration.csv")
surficial_path = Path("/Users/huiyaokuang/OneDrive/Erdos Institute/radon project/data/surficial geology/2014 surficial/cgm_0195_p1/Data/SHP/Surficial/GEO_POLYS.shp")
fsa_path = Path("/Users/huiyaokuang/OneDrive/Erdos Institute/radon project/data/FSA data/gfsa000a11a_e/gfsa000a11a_e.shp")
radon_path = Path("/Users/huiyaokuang/OneDrive/Erdos Institute/radon project/data/cross Canada radon survey/radon-concentration.csv")

### 1. Surficial data

##### Data understanding

In [3]:
surf = gpd.read_file(surficial_path)
surf["UTYPE1"].value_counts()

UTYPE1
Glacial sediments - Veneer                                         2443
Bedrock - Undifferentiated                                         2059
Glacial sediments - Blanket                                        1872
Glaciolacustrine sediments - Offshore sediments                     741
Glaciofluvial sediments - Outwash plain sediments                   637
Glaciomarine sediments - Littoral and nearshore sediments           505
Glaciofluvial sediments - Ice-contact sediments                     467
Glaciomarine sediments - Offshore sediments                         404
Glacial Ice or Snowpack - Snowpacks                                 360
Glacial sediments - Moraine complex                                 359
Colluvial and mass-wasting deposits - Undifferentiated deposits     354
Alluvial sediments - Undifferentiated sediments                     323
Glaciolacustrine sediments - Littoral and nearshore sediments       320
Glacial sediments - Hummocky till                        

In [4]:
surf.columns

Index(['ULABEL1', 'UTYPE1', 'USUBCAT1', 'URELATION', 'ULABEL2', 'UTYPE2',
       'USUBCAT2', 'EVENT1', 'EVENT2', 'REMARKS', 'SYMBOL1', 'SOURCE',
       'ORIG_CODE', 'HYDRO_INT', 'geometry'],
      dtype='str')

In [5]:
surf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 12385 entries, 0 to 12384
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   ULABEL1    12385 non-null  str     
 1   UTYPE1     12385 non-null  str     
 2   USUBCAT1   12385 non-null  str     
 3   URELATION  12385 non-null  str     
 4   ULABEL2    0 non-null      object  
 5   UTYPE2     0 non-null      object  
 6   USUBCAT2   12378 non-null  str     
 7   EVENT1     0 non-null      object  
 8   EVENT2     0 non-null      object  
 9   REMARKS    1719 non-null   str     
 10  SYMBOL1    12385 non-null  str     
 11  SOURCE     12385 non-null  str     
 12  ORIG_CODE  0 non-null      object  
 13  HYDRO_INT  12385 non-null  str     
 14  geometry   12385 non-null  geometry
dtypes: geometry(1), object(5), str(9)
memory usage: 2.5+ MB


In [6]:
surf["ULABEL1"]

0         Lo
1         Ln
2        GLn
3        GLn
4        GFp
        ... 
12380      R
12381      R
12382      R
12383      R
12384    GLo
Name: ULABEL1, Length: 12385, dtype: str

In [7]:
surf["UTYPE1"]

0                Lacustrine sediments - Offshore sediments
1        Lacustrine sediments - Littoral and nearshore ...
2        Glaciolacustrine sediments - Littoral and near...
3        Glaciolacustrine sediments - Littoral and near...
4        Glaciofluvial sediments - Outwash plain sediments
                               ...                        
12380                           Bedrock - Undifferentiated
12381                           Bedrock - Undifferentiated
12382                           Bedrock - Undifferentiated
12383                           Bedrock - Undifferentiated
12384      Glaciolacustrine sediments - Offshore sediments
Name: UTYPE1, Length: 12385, dtype: str

In [8]:
surf["USUBCAT1"].value_counts()

USUBCAT1
Not applicable    7434
Unspecified       4951
Name: count, dtype: int64

In [9]:
surf["URELATION"].value_counts()

URELATION
None    12385
Name: count, dtype: int64

In [10]:
surf["USUBCAT2"].value_counts()

USUBCAT2
Not applicable    12378
Name: count, dtype: int64

In [11]:
surf["REMARKS"].value_counts()

REMARKS
GLb can be GLb or GLo          741
GMb can be GMb or GMo          404
end and interlobate moraine    358
Mb can be Mb or Mo             178
Lb can be Lb or Lo              37
water                            1
Name: count, dtype: int64

In [12]:
surf["SYMBOL1"].value_counts()

SYMBOL1
3.01.10.355    2443
3.01.13.185    2059
3.01.10.359    1872
3.01.08.637     741
3.01.07.245     637
3.01.09.513     505
3.01.07.217     467
3.01.09.519     404
3.01.15.002     360
3.01.10.377     359
3.01.01.152     354
3.01.04.263     323
3.01.08.612     320
3.01.10.375     277
3.01.02.012     250
3.01.09.483     244
3.01.06.509     177
3.01.01.092     176
3.01.11.177     143
3.01.06.493      70
3.01.11.175      52
3.01.05.573      50
3.01.03.297      50
3.01.05.577      37
3.01.16.707      15
Name: count, dtype: int64

In [13]:
# Mapping of the coding to major sediment type (the 3rd number in SYMBOL1). Reference: /StylesFonts/GSC_SurficialStyleChart.pdf

major_sedi_type_mapping = {
    "10": "Glacial sediments",
    "13": "Bedrock",
    "08": "Glaciolacustrine sediments",
    "07": "Glaciofluvial sediments",
    "09": "Glaciomarine sediments", 
    "15": "Glacier ice or snowpack",
    "01": "Colluvial and Mass-wasting deposits",
    "04": "Alluvial sediments",
    "02": "Organic deposits",
    "06": "Marine sediments",
    "11": "Weathered bedrock or regolith",
    "05": "Lacustrine sediments",
    "03": "Eolian sediments",
    "16": "Volcanic deposits"
}

In [14]:
surf["major_sedi_code"] = surf["SYMBOL1"].str.split(".").str[2]
surf["major_sedi_type1"] = surf["major_sedi_code"].map(major_sedi_type_mapping)

In [15]:
surf["SOURCE"].value_counts()

SOURCE
Map 123    12385
Name: count, dtype: int64

In [16]:
surf["HYDRO_INT"].value_counts()

HYDRO_INT
Land     8448
Water    3937
Name: count, dtype: int64

In [17]:
surf["geometry"].value_counts()

geometry
MULTIPOLYGON Z (((1059176.875 -692868.062 0, 1...    1
POLYGON Z ((1061234.875 -694253.875 0, 1060588...    1
POLYGON Z ((1032545.496 -693150.165 0, 1030351...    1
POLYGON Z ((1063728.761 -676828.749 0, 1062204...    1
POLYGON Z ((1055660 -676224 0, 1047434 -682153...    1
                                                    ..
POLYGON Z ((853576.865 1390131.736 0, 853586.3...    1
MULTIPOLYGON Z (((861475.063 1401139.375 0, 86...    1
MULTIPOLYGON Z (((840383.502 1596242.168 0, 84...    1
POLYGON Z ((108042.399 3015115.75 0, 107498.48...    1
POLYGON Z ((627672.188 12497.373 0, 629798.181...    1
Name: count, Length: 12385, dtype: int64

In [18]:
surf.crs

<Compound CRS: COMPD_CS["Clarke_1866_Lambert_Conformal_Conic + CG ...>
Name: Clarke_1866_Lambert_Conformal_Conic + CGVD28 height
Axis Info [cartesian|vertical]:
- [east]: Easting (metre)
- [north]: Northing (metre)
- [up]: Gravity-related height (metre)
Area of Use:
- undefined
Datum: Not specified (based on Clarke 1866 ellipsoid)
- Ellipsoid: Clarke 1866
- Prime Meridian: Greenwich
Sub CRS:
- Clarke_1866_Lambert_Conformal_Conic
- CGVD28 height

In [19]:
surf["geometry"].value_counts()

geometry
MULTIPOLYGON Z (((1059176.875 -692868.062 0, 1...    1
POLYGON Z ((1061234.875 -694253.875 0, 1060588...    1
POLYGON Z ((1032545.496 -693150.165 0, 1030351...    1
POLYGON Z ((1063728.761 -676828.749 0, 1062204...    1
POLYGON Z ((1055660 -676224 0, 1047434 -682153...    1
                                                    ..
POLYGON Z ((853576.865 1390131.736 0, 853586.3...    1
MULTIPOLYGON Z (((861475.063 1401139.375 0, 86...    1
MULTIPOLYGON Z (((840383.502 1596242.168 0, 84...    1
POLYGON Z ((108042.399 3015115.75 0, 107498.48...    1
POLYGON Z ((627672.188 12497.373 0, 629798.181...    1
Name: count, Length: 12385, dtype: int64

##### Data cleaning

In [20]:
#surf = surf.to_crs("EPSG:4269")

In [21]:
#surf.crs

In [22]:
#surf["UTYPE1"].value_counts()

In [23]:
#surf["major_sedi_type1"].value_counts()

# UTYPE1 and SYMBOL1 correspond with each other. Use UTYPE1.


In [24]:
surf["major_sedi_type"] = surf["UTYPE1"].str.split(" -").str[0]
cols = ["major_sedi_type", "ULABEL1", "HYDRO_INT", "geometry"]
surf_df = surf[cols].copy()
surf_df

,major_sedi_type,ULABEL1,HYDRO_INT,geometry
0,Lacustrine sediments,Lo,Land,"MULTIPOLYGON Z (((1059176.875 -692868.062 0, 1..."
1,Lacustrine sediments,Ln,Land,"POLYGON Z ((1061234.875 -694253.875 0, 1060588..."
2,Glaciolacustrine sediments,GLn,Land,"POLYGON Z ((1032545.496 -693150.165 0, 1030351..."
3,Glaciolacustrine sediments,GLn,Land,"POLYGON Z ((1063728.761 -676828.749 0, 1062204..."
4,Glaciofluvial sediments,GFp,Land,"POLYGON Z ((1055660 -676224 0, 1047434 -682153..."
...,...,...,...,...
12380,Bedrock,R,Water,"POLYGON Z ((853576.865 1390131.736 0, 853586.3..."
12381,Bedrock,R,Water,"MULTIPOLYGON Z (((861475.063 1401139.375 0, 86..."
12382,Bedrock,R,Water,"MULTIPOLYGON Z (((840383.502 1596242.168 0, 84..."
12383,Bedrock,R,Water,"POLYGON Z ((108042.399 3015115.75 0, 107498.48..."


In [25]:
surf_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 12385 entries, 0 to 12384
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   major_sedi_type  12385 non-null  object  
 1   ULABEL1          12385 non-null  str     
 2   HYDRO_INT        12385 non-null  str     
 3   geometry         12385 non-null  geometry
dtypes: geometry(1), object(1), str(2)
memory usage: 463.3+ KB


### 2. Join FSA data

In [26]:
fsa_gdf = gpd.read_file(fsa_path)

In [27]:
# Convert CRS to Canada Albers (meters) - to calculate the area of sediment types
surf_proj = surf[["major_sedi_type", "geometry"]].to_crs("EPSG:3347")
fsa_proj  = fsa_gdf[["CFSAUID", "geometry"]].to_crs("EPSG:3347")

In [28]:
surf_proj

,major_sedi_type,geometry
0,Lacustrine sediments,"MULTIPOLYGON Z (((6994566.827 690443.449 0, 69..."
1,Lacustrine sediments,"POLYGON Z ((6996554.049 688957.945 0, 6995850...."
2,Glaciolacustrine sediments,"POLYGON Z ((6967954.537 691473.234 0, 6965691...."
3,Glaciolacustrine sediments,"POLYGON Z ((6999902.763 706239.353 0, 6998269...."
4,Glaciofluvial sediments,"POLYGON Z ((6991873.769 707240.75 0, 6983366.0..."
...,...,...
12380,Bedrock,"POLYGON Z ((6891779.865 2781030.902 0, 6891789..."
12381,Bedrock,"MULTIPOLYGON Z (((6900210.147 2791635.996 0, 6..."
12382,Bedrock,"MULTIPOLYGON Z (((6888751.136 2987535.984 0, 6..."
12383,Bedrock,"POLYGON Z ((6227192.931 4440700.331 0, 6226670..."


Overlay area and calculate the percentage of sedimentary types in each FSA

In [29]:
geo_int = gpd.overlay(surf_proj, fsa_proj, how="intersection").rename(columns={"CFSAUID":"FSA"})
geo_int["area"] = geo_int.geometry.area
fsa_type_area = geo_int.groupby(["FSA","major_sedi_type"])["area"].sum().reset_index()
X_area = fsa_type_area.pivot(index="FSA", columns="major_sedi_type", values="area").fillna(0)
X_perc = X_area.div(X_area.sum(axis=1), axis=0)
X_perc.columns = [f"sedi_{c}" for c in X_perc.columns]

In [30]:
X_perc

,sedi_Alluvial sediments,sedi_Bedrock,sedi_Colluvial and mass-wasting deposits,sedi_Eolian sediments,sedi_Glacial Ice or Snowpack,sedi_Glacial sediments,sedi_Glaciofluvial sediments,sedi_Glaciolacustrine sediments,sedi_Glaciomarine sediments,sedi_Lacustrine sediments,sedi_Marine sediments,sedi_Organic deposits,sedi_Volcanic deposits,sedi_Weathered bedrock or regolith
FSA,,,,,,,,,,,,,,
A0A,0.000000,0.086583,0.000000,0.0,0.00000,0.883009,0.000000,0.000000,0.000000,0.000000,0.030408,0.000000,0.000000,0.0
A0B,0.000000,0.102177,0.000000,0.0,0.00000,0.825744,0.000000,0.000000,0.000000,0.000000,0.072079,0.000000,0.000000,0.0
A0C,0.000000,0.153760,0.000000,0.0,0.00000,0.846240,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
A0E,0.000000,0.181130,0.000000,0.0,0.00000,0.571070,0.000000,0.000000,0.122135,0.000000,0.125665,0.000000,0.000000,0.0
A0G,0.000000,0.027193,0.000000,0.0,0.00000,0.870902,0.000000,0.000000,0.008905,0.000000,0.093000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
X0G,0.405759,0.000000,0.000000,0.0,0.00000,0.594241,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
X1A,0.000000,0.083868,0.000000,0.0,0.00000,0.908138,0.007994,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
Y0A,0.006626,0.273048,0.030140,0.0,0.00000,0.645517,0.030126,0.014543,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


### 3. Join radon survey data

In [31]:
radon_df = pd.read_csv(radon_path)

In [32]:
print(radon_df.shape)
radon_df = radon_df.drop(
    columns=[
        "Unnamed: 7", "Unnamed: 8", "Unnamed: 9","Unnamed: 10",
        "Unnamed: 11","Unnamed: 12","Health Region2007","HealthRegionCode2007","ResultNumber"
        ]
    ).copy()
radon_df.head(3)

(13815, 13)


,ProvinceTerritory,ForwardSortationAreaCodes,TestDurationInDays,AverageRadonConcentrationInBqPerM3
0,NL,A0A,127.0,20
1,NL,A0A,108.0,36
2,NL,A0E,91.0,<15


Replace "<15" in Average Radon Concentration with 7.5

In [33]:
radon_df["AverageRadonConcentrationInBqPerM3"] = radon_df["AverageRadonConcentrationInBqPerM3"].replace("<15", 7.5).astype(float)

In [34]:
# Rename radon_df columns
radon_df = radon_df.rename(columns={"ForwardSortationAreaCodes":"FSA"})

In [35]:
print(radon_df["FSA"].value_counts())  # "*" means some data has been removed in order to protect the identity of study participants.
radon_df = radon_df[radon_df["FSA"] != "*"].copy()

FSA
*      247
Y1A    179
T0H    175
G8P    143
S0J    136
      ... 
V7R      2
V8N      2
V8R      2
V8X      2
V9S      2
Name: count, Length: 1015, dtype: int64


In [36]:
# radon_df group by FSA
#radon_fsa = radon_df.groupby("FSA").agg(
#    mean_radon = ("AverageRadonConcentrationInBqPerM3","mean"),
#    median_radon = ("AverageRadonConcentrationInBqPerM3","median"),
#    mean_duration = ("TestDurationInDays","mean"),
#    n_tests = ("AverageRadonConcentrationInBqPerM3","count"),
#    Province = ("ProvinceTerritory", "first")
#).reset_index()

In [37]:
#radon_fsa.isna().sum()

Merge radon_df with surf_fsa

In [38]:
df_fsa = X_perc.reset_index().merge(radon_df, on="FSA", how="inner")
#df_model = df_fsa.dropna(subset=["mean_radon"]).copy()

In [39]:
df_fsa

,FSA,sedi_Alluvial sediments,sedi_Bedrock,sedi_Colluvial and mass-wasting deposits,sedi_Eolian sediments,sedi_Glacial Ice or Snowpack,sedi_Glacial sediments,sedi_Glaciofluvial sediments,sedi_Glaciolacustrine sediments,sedi_Glaciomarine sediments,sedi_Lacustrine sediments,sedi_Marine sediments,sedi_Organic deposits,sedi_Volcanic deposits,sedi_Weathered bedrock or regolith,ProvinceTerritory,TestDurationInDays,AverageRadonConcentrationInBqPerM3
0,A0A,0.0,0.086583,0.000000,0.0,0.0,0.883009,0.000000,0.0000,0.0,0.0,0.030408,0.0,0.0,0.0,NL,127.0,20.0
1,A0A,0.0,0.086583,0.000000,0.0,0.0,0.883009,0.000000,0.0000,0.0,0.0,0.030408,0.0,0.0,0.0,NL,108.0,36.0
2,A0A,0.0,0.086583,0.000000,0.0,0.0,0.883009,0.000000,0.0000,0.0,0.0,0.030408,0.0,0.0,0.0,NL,91.0,31.0
3,A0A,0.0,0.086583,0.000000,0.0,0.0,0.883009,0.000000,0.0000,0.0,0.0,0.030408,0.0,0.0,0.0,NL,94.0,7.5
4,A0A,0.0,0.086583,0.000000,0.0,0.0,0.883009,0.000000,0.0000,0.0,0.0,0.030408,0.0,0.0,0.0,NL,92.0,7.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13562,Y1A,0.0,0.174265,0.034157,0.0,0.0,0.657710,0.042669,0.0912,0.0,0.0,0.000000,0.0,0.0,0.0,YT,101.0,1375.0
13563,Y1A,0.0,0.174265,0.034157,0.0,0.0,0.657710,0.042669,0.0912,0.0,0.0,0.000000,0.0,0.0,0.0,YT,91.0,1530.0
13564,Y1A,0.0,0.174265,0.034157,0.0,0.0,0.657710,0.042669,0.0912,0.0,0.0,0.000000,0.0,0.0,0.0,YT,93.0,1623.0
13565,Y1A,0.0,0.174265,0.034157,0.0,0.0,0.657710,0.042669,0.0912,0.0,0.0,0.000000,0.0,0.0,0.0,YT,110.0,2281.0


In [40]:
# Rename columns
df_fsa.columns = (
    df_fsa.columns
    .str.lower()
    .str.replace(" ", "_")
)

In [41]:
df_fsa.info()

<class 'pandas.DataFrame'>
RangeIndex: 13567 entries, 0 to 13566
Data columns (total 18 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   fsa                                       13567 non-null  str    
 1   sedi_alluvial_sediments                   13567 non-null  float64
 2   sedi_bedrock                              13567 non-null  float64
 3   sedi_colluvial_and_mass-wasting_deposits  13567 non-null  float64
 4   sedi_eolian_sediments                     13567 non-null  float64
 5   sedi_glacial_ice_or_snowpack              13567 non-null  float64
 6   sedi_glacial_sediments                    13567 non-null  float64
 7   sedi_glaciofluvial_sediments              13567 non-null  float64
 8   sedi_glaciolacustrine_sediments           13567 non-null  float64
 9   sedi_glaciomarine_sediments               13567 non-null  float64
 10  sedi_lacustrine_sediments                 135

Save df_fsa to .csv

In [45]:
df_fsa.to_csv("surficial_fsa_radon.csv", index=False)

In [43]:
import sys
print(sys.executable)

/opt/anaconda3/envs/erdos_ds_environment/bin/python


In [44]:
import geopandas as gpd
gpd.__file__

'/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/geopandas/__init__.py'